### Notebook to find best Selector-PI design (part of Biogoals.Select tool)

**Authors:** Davide Carecci  
**Created:** May 2025  
**Last Modified:** November 05, 2025  
**Version:** 1.1  
**Institution:** Politecnico di Milano  

---

#### Revision History

| Date       | Version | Author(s)              | Description                                                                                                    |
|------------|---------|------------------------|----------------------------------------------------------------------------------------------------------------|
| 2025-02-18 | 1.0     | D. Carecci             | Initial implementation for the simulation study carried out at SECO (UMONS, Mons, Belgium). NMPC vs selectorPI |
| 2025-11-05 | 1.1     | D. Carecci             | Code and folder cleaning and documentation for handle over a copy to A2A S.p.A                                 |

---
_Note_: the code for the optimization of the selectorPI design here presented is contextualized in the example of the simulation study published at [copy link IEEE_CDC2025]() i.e.
    NMPC vs selectorPI (T_c = 6 hours for both) to control the agri-AcoDM model with challanging artificial disturbance (see manuscript for details).
    The artificial disturbance must be not considered when running the present code (change setting directly inside the Modelica library of agri-AcoDM).
    A (first) version for the UIT experimental activity (31.01.2024) and the work published at [IEEE_CDC2024](https://ieeexplore.ieee.org/document/10886344) also exists. Contact the authors to request a copy.


In [ ]:
# IMPORT THE MAIN PACKAGES NEEDED FOR THE PROGRAM TO RUN
# Standard packages
import numpy as np
import pylab as pl;
from scipy.optimize import minimize
from scipy.optimize import differential_evolution, NonlinearConstraint, Bounds
import subprocess
import DyMat
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import logging
import os
# Custom packages
from general_utils.modelica_integrator import modelica_integrator
from general_utils.modelica_optimizer import modelica_optimizer, exponential_penalty, min_error_constrained
from general_utils.read_file import read_excel_file

In [ ]:
# Configure logger to print info messages in console
logging.basicConfig(level=logging.INFO)
# Set current working directory where user wants outputs to be saved
# In this folder all the files created by OMC to integrate modelica models will be stored
os.chdir("C:/Users/lenovo/OneDrive - Politecnico di Milano/Work_cloud/DOTTORATO/Sperimentazione UIT/SelectorPI controller/Private/")

# RUN NOMINAL SIMULATION WITH MODELICA TO EXTRACT NOMINAL QUANTITIES
# Set simulation options
# Modelica model file path and name
FilePath = "C:/Users/lenovo/OneDrive - Politecnico di Milano/Work_cloud/DOTTORATO/Modelling/Pilot_plant/ADM1_X_14_03_2024_batch_GQ - Copia_DC.mo"
ModelName = "ADM1_P.UIT_CL_oldselector_newmodel"
folder_name = "selector_6h" # Where to save outputs of the simulations
# Simulation parameters
interval = 3600
stoptime = interval*np.ceil(5.184e+06/interval)
tolerance = '1e-5'
# Outputs to extract from the simulation (names and internal Modelica paths)
outputs_extract_names = ["y_gas[4]", "TVFA", "COD_tot",
                        "y1_set", "switch.y", "follower.PV", "y2_set.y",
                        "follower.e","follower.uio", "follower.ui", "follower.u",
                        "header.e","header.uio", "header.ui", "header.u"
                        ]
path_to_outputs_extract=["uit_ol.uit", "uit_ol.uit.digester", "uit_ol.uit.digester",
                             "", "", "", "", 
                             "","", "", "",
                             "","", "", "",
                             ]

In [ ]:
# RUN NOMINAL (as it is in the OpenModelica implementation) SIMULATION WITH MODELICA TO EXTRACT NOMINAL (initial guesses) QUANTITIES
y_nom, final_states_nom, mat_file_nom = modelica_integrator(
    FilePath,
    ModelName,
    folder_name,
    {},
    {},
    {},
    interval,
    0,
    stoptime,
    tolerance,
    x0_extract_names=[],
    path_to_x0_extract=[],
    path_to_outputs_extract=path_to_outputs_extract,
    outputs_extract_names = outputs_extract_names,
    log=True,
    return_dymat = True # To return also the dymat object with all simulation results (to extract values of parameters/constants that cannot be extracted directly as outputs)
    )

In [ ]:
# Extract other nominal quantities from the .mat file
# x are the decision variables i.e. the PI controller parameters
x_names = ["kp1","kp2","Ti1","Ti2"]
x_nom = [mat_file_nom.data(f'{x_name}')[-1] for x_name in x_names] 
# Note: initial values before optimization were: [1e-2, 1e-2, 1e5, 1e5]
# x_other_nom = [mat_file_nom.data("hysteresisComparator.uLow"), # Uncomment just for checking and plotting purposes
#                mat_file_nom.data("hysteresisComparator.uHigh")]
# Define scaling factors for decision variables
x_scales = [1e2, 1e2, 1e-5, 1e-5]
# Scale nominal decision variables within 1-10 range to facilitate optimization (x_nom_scaled)
x_nom_scaled = [x*s for x,s in zip(x_nom, x_scales)]

# Create dict from x_names and x_scales
x_scale_dict = dict(zip(x_names, x_scales))
x0_dict = dict(zip(x_names, x_nom_scaled))
# Print to check 
print(f"Actual nominal values are: {x_nom}\n"
f"Scaled values are: {x_nom_scaled}")

In [ ]:
# NOMINAL: compute objective functions
# Extract values from the dataframe of output results
# DEFINE TARGET TRAJECTORY DATAFRAME
target_df = y_nom.copy()
target_df = target_df[['Timestamp', 'y1_set']]
target_df.rename(columns={'y1_set':'y_gas[4]'}, inplace=True) # If using min_error and min_error_constrained function, rename to match output name. Contrary if using cumulative_error function!
# Minimize distance of methane flow rate from setpoint
f_obj_maxim_traj_nom = ((y_nom['y1_set'].values - y_nom['y_gas[4]'].values)/y_nom['y1_set'].values)**2
f_obj_maxim_nom = f_obj_maxim_traj_nom.sum()/len(y_nom['y_gas[4]'].values)

In [ ]:
# DECLARE CONSTRAINTS
# Declare bounds for contraints
VFA_ub = 1600 # Upper bound of TVFA (mg_{COD} L^{-1}). A bit higher with respect to the one of the feed optimization (...!!!!!) to allow the nominal simulation to result without penalties
CODout_ub = 60 # Upper bound of COD in the digestate (g_{COD} L^{-1})

# Check if nominal simulation overcomes the constraints and apply penalties to the objective functions if so
# Note: in version 1.1 the penalties are not multiplicative factors to the objective function, but added terms!!
# CODout upper bound
if y_nom['COD_tot'].values[-1] > CODout_ub:
    COD_penalty = exponential_penalty(y_nom['COD_tot'].values[-1], -np.inf, CODout_ub, 1e-2)
    f_obj_maxim_nom = f_obj_maxim_nom*(1+COD_penalty)
    print(f'CODout ub is overcome by {y_nom["COD_tot"].values[-1]-CODout_ub}: f_obj_maxim increasd by {COD_penalty*100}%')

# TVFA upper bound
# Must be lower than the peak Haldane value inside agri-AcoDM
if y_nom['TVFA'].values.max() > VFA_ub:
    VFA_penalty = exponential_penalty(y_nom['TVFA'].values.max(), -np.inf, VFA_ub, 1e-3)
    f_obj_maxim_nom = f_obj_maxim_nom*(1+VFA_penalty) 
    print(f'VFAmax ub is overcome by {y_nom["TVFA"].values.max()-VFA_ub}: f_obj_maxim increasd by {VFA_penalty*100}%')
    
# Reformulate the constraints in the format required by the optimizer
constraints = [
    {
        'type': 'custom',
        'name': 'COD_tot_max_cap',                  # COD_tot upper bound      
        'fn': lambda df: df['COD_tot'].iloc[-1],    # scalar metric
        'upper': CODout_ub,                         # upper bound
        'weight': 50,                               # penalty strength
        'growth_rate': 2.0
    },
    {
        'type': 'custom',
        'name': 'TVFA_max_cap',             # TVFA upper bound
        'fn': lambda df: df['TVFA'].max(),  # scalar metric
        'upper': VFA_ub,                    # upper bound
        'weight': 50,                       # penalty strength
        'growth_rate': 2.0
    }
]

In [ ]:
# Plot just for checking purposes
# pl.plot(y_nom['TVFA'].values)
pl.plot(y_nom['y1_set'].values)
pl.plot(y_nom['y_gas[4]'].values)
pl.plot(f_obj_maxim_traj_nom)

In [ ]:
# RUN OPTIMIZTION WITH MODELICA INTEGRATOR AND OPTIMIZER
# Modify k_P and T_I to minimize distance of methane flow rate from setpoint while keeping TVFA and COD within bounds
# Set parameter bounds
param_bounds={x_names[0]: (0.15*x_nom_scaled[0], 6*x_nom_scaled[0]),
                x_names[1]: (0.15*x_nom_scaled[1], 6*x_nom_scaled[1]),
                x_names[2]: (0.15*x_nom_scaled[2], 6*x_nom_scaled[2]),
                x_names[3]: (0.15*x_nom_scaled[3], 6*x_nom_scaled[3])}
#Start parameter values within 1-10
#Remember to rescale outside Nelder-Mead (as in Marco's script) and within TNC ("rescale" solver option)
x0 = np.array(x_nom_scaled)

In [ ]:
# DECLARE OPTIMIZER
optimizer = modelica_optimizer(min_error_constrained, # Cost function to minimize
                               initial_guesses= x0, # Use as initial guesses the nominal scaled values
                               param_bounds=param_bounds, # Define parameter bounds for exploration
                               cost_args={'target_values': target_df, # Arguments of the cost function: target values for the optimization
                                          'constraints': constraints}, # Arguments of the cost function: constraints to be enforced
                               integrator_kwargs={ # Arguments for the Modelica integrator
                                       'mo_path':FilePath,
                                        'model_name':ModelName,
                                        'folder_name':folder_name,
                                        "param_scale_dict":x_scale_dict,
                                        'x0_dict':{}, # Do not confuse: this are the initial conditons for the integrator, not the decision variables!
                                        'time_interval':interval,
                                        'start_time':0,
                                        'stop_time':stoptime,
                                        'tolerance':tolerance,
                                        'x0_extract_names':[],
                                        'path_to_x0_extract':[],
                                        'path_to_outputs_extract':path_to_outputs_extract,
                                        'outputs_extract_names':outputs_extract_names,
                                        'log': True,
                                        'return_dymat': False},
                                        )

In [ ]:
# RUN OPTIMIZATION
# In this example, differential evolution is used with max 10 iterations
# Check the other default optimization options in the modelica_optimizer class (e.g. number of workers, population size, mutation factor, recombination rate, tolerance for stop criteria etc.)
result = optimizer.differential_evolution(max_iter=10)

In [ ]:
# Save the DataFrames to an Excel file
# Get the current date to create unique filename
current_date = datetime.now().date()
formatted_date = current_date.strftime('%d.%m.%Y')
excel_file_path = f'C:/Users/lenovo/OneDrive - Politecnico di Milano/Work_cloud/DOTTORATO/Sperimentazione UIT/SelectorPI controller/DE_optim_feval_{formatted_date}_tuning.xlsx'
# optimizer.iter_df contains the main results for each function evaluation...
optimizer.save_iter_df(excel_file_path)

In [ ]:
# Note that optimizer.y_df is the last simulation output dataframe
# Not necessarily the one corresponding to the best solution found
pl.plot(optimizer.y_df['y1_set'].values)
pl.plot(optimizer.y_df['y_gas[4]'].values)

In [ ]:
# In case optimal parameters have been found already, just for error computation purposes
# Re-declare optimal parameter
if 'result' in locals(): # If optimizer has converged and result object exists
    x = result.x
    optimal_x = [xi/si for xi, si in zip(x, x_scales)] # Scale back to original values
else: # Extract parameters that lead to minimum objective function value from optimizer.iter_df
    min_row = optimizer.iter_df.loc[optimizer.iter_df['total_cost'].idxmin()]
    optimal_x = min_row[x_names].values # Scale back to original values
    x = [xi*si for xi, si in zip(optimal_x, x_scales)] # Scaled values
print(f"Optimal parameters are: {dict(zip(x_names, optimal_x))}")

In [ ]:
# RUN OPTIMIZED (with results found above) SIMULATION WITH MODELICA TO CHECK AND PLOT QUANTITIES
y_opt, final_states_opt, mat_file_opt = modelica_integrator(
    FilePath,
    ModelName,
    folder_name,
    dict(zip(x_names, x)), # Now enforce optimal parameters found into the Modelica simulation
    x_scale_dict,
    {},
    interval,
    0,
    stoptime,
    tolerance,
    x0_extract_names=[],
    path_to_x0_extract=[],
    path_to_outputs_extract=path_to_outputs_extract,
    outputs_extract_names = outputs_extract_names,
    log=True,
    return_dymat = True
    )

# Compute the objective function
# Minimize distance of methane flow rate from setpoint
f_obj_maxim_traj_opt = ((y_opt['y1_set'].values - y_opt['y_gas[4]'].values)/y_opt['y1_set'].values)**2
f_obj_maxim_opt = f_obj_maxim_traj_opt.sum()

# CONSTRAINTS
# CODout upper bound
if y_opt['COD_tot'].values[-1] > CODout_ub:
    COD_penalty = exponential_penalty(y_opt['COD_tot'].values[-1], -np.inf, CODout_ub, 1e-2)
    f_obj_maxim_opt = f_obj_maxim_opt*(1+COD_penalty)
    print(f'CODout ub is overcome by {y_opt["COD_tot"].values[-1]-CODout_ub}: f_obj_maxim increasd by {COD_penalty*100}%')

# TVFA upper bound
# Must be lower than the peak Haldane value inside agri-AcoDM
if y_opt['TVFA'].values.max() > VFA_ub:
    VFA_penalty = exponential_penalty(y_opt['TVFA'].values.max(), -np.inf, VFA_ub, 1e-3)
    f_obj_maxim_opt = f_obj_maxim_opt*(1+VFA_penalty)
    print(f'VFAmax ub is overcome by {y_opt["TVFA"].values.max()-VFA_ub}: f_obj_maxim increasd by {VFA_penalty*100}%')

In [ ]:
# Plot and compare tracking performance of methane flow rate
pl.figure(figsize=(10, 4))
pl.plot(y_nom['y_gas[4]'],color='r')
pl.plot(y_opt['y_gas[4]'],color='b')
pl.plot(y_nom['y1_set'],color='g')

In [ ]:
# Plot and compare manipulated variable of the follower controller (proportional and integral actions)
pl.figure(figsize=(10, 4))
pl.plot(y_nom['follower.ui'],color='r')
pl.plot(y_opt['follower.ui'],color='b')
pl.plot(y_nom['follower.u']-y_nom['follower.ui'],color='r',linestyle='--')
pl.plot(y_opt['follower.u']-y_opt['follower.ui'],color='b',linestyle='--')

In [ ]:
# Plot and compare selector output (switching signal) i.e. the actual control action fed to the plant
pl.figure(figsize=(10, 4))
pl.plot(y_nom['switch.y'],color='r')
pl.plot(y_opt['switch.y'],color='b')

In [ ]:
# Plot and compare TVFA profiles
pl.figure(figsize=(10, 4))
pl.plot(y_nom['TVFA'],color='r')
pl.plot(y_opt['TVFA'],color='b')